In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

SOURCE_URL = "https://data.open-power-system-data.org/time_series/2020-10-06/time_series_60min_singleindex.csv"
start, train_end, validation_end, end = map(pd.Timestamp, ["2015-01-01T00:00:00Z", "2019-01-01T00:00:00Z", "2020-01-01T00:00:00Z", "2020-10-01T00:00:00Z"])
raw = pd.read_csv(SOURCE_URL, usecols=["utc_timestamp", "DE_load_actual_entsoe_transparency"])
raw = raw.rename(columns={"utc_timestamp": "timestamp", "DE_load_actual_entsoe_transparency": "load"})
raw["timestamp"] = pd.to_datetime(raw["timestamp"], utc=True, errors="coerce")
raw["load"] = pd.to_numeric(raw["load"], errors="coerce")
raw = raw.loc[(raw.timestamp >= start) & (raw.timestamp < end)].copy()
duplicate_rows = int(raw.duplicated("timestamp").sum())
missing_load = int(raw.load.isna().sum())
invalid_load = int((raw.load.notna() & (~np.isfinite(raw.load) | (raw.load <= 0))).sum())
# Match the project loader: keep the last duplicate, hourly mean, causal forward fill up to 3 hours.
series = (raw.dropna(subset=["timestamp", "load"]).sort_values("timestamp")
          .drop_duplicates("timestamp", keep="last").set_index("timestamp")["load"].astype(float)
          .resample("h").mean().ffill(limit=3))
if series.isna().any():
    raise ValueError(f"Unresolved missing hourly values: {int(series.isna().sum())}")
if (series <= 0).any() or not np.isfinite(series.to_numpy()).all():
    raise ValueError("Non-positive or non-finite load remains")
if series.index[0] != start or series.index[-1] != end - pd.Timedelta(hours=1):
    raise ValueError(f"Unexpected coverage: {series.index[0]} to {series.index[-1]}")
train, validation, test = series.loc[series.index < train_end], series.loc[(series.index >= train_end) & (series.index < validation_end)], series.loc[series.index >= validation_end]
out = series.rename("load").rename_axis("timestamp").reset_index()
out["timestamp"] = out.timestamp.dt.strftime("%Y-%m-%dT%H:%M:%SZ")
output_path = Path("/kaggle/working/germany_hourly_load.csv")
out.to_csv(output_path, index=False)
print({"source_version":"2020-10-06", "series":"DE_load_actual_entsoe_transparency", "units":"MW", "timezone":"UTC", "rows":len(series), "first":str(series.index[0]), "last":str(series.index[-1]), "duplicates":duplicate_rows, "missing_load_in_selected_period":missing_load, "invalid_load":invalid_load, "unresolved_missing_hours":0, "train_hours":len(train), "validation_hours":len(validation), "test_hours_reserved":len(test), "csv":str(output_path)})

{'source_version': '2020-10-06', 'series': 'DE_load_actual_entsoe_transparency', 'units': 'MW', 'timezone': 'UTC', 'rows': 50400, 'first': '2015-01-01 00:00:00+00:00', 'last': '2020-09-30 23:00:00+00:00', 'duplicates': 0, 'missing_load_in_selected_period': 0, 'invalid_load': 0, 'unresolved_missing_hours': 0, 'train_hours': 35064, 'validation_hours': 8760, 'test_hours_reserved': 6576, 'csv': '/kaggle/working/germany_hourly_load.csv'}


In [2]:
import transformers, peft, torch
from transformers import TimesFm2_5ModelForPrediction
print({"transformers": transformers.__version__, "peft": peft.__version__, "torch": torch.__version__, "cuda_devices": torch.cuda.device_count(), "timesfm_class": TimesFm2_5ModelForPrediction.__name__})

{'transformers': '5.20.2', 'peft': '0.21.2', 'torch': '2.11.0+cu130', 'cuda_devices': 2, 'timesfm_class': 'TimesFm2_5ModelForPrediction'}


In [3]:
%pip install --upgrade torchao
from pathlib import Path
import subprocess, sys, json, gc
import numpy as np
import pandas as pd
import torch

REPO = Path("/kaggle/working/energy-demand-timesfm")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gufrxn999/energy-demand-timesfm.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from src.models import TimesFMZeroShot, seasonal_naive
from src.metrics import score
ADAPTER = REPO / "outputs/timesfm-lora-pjme-t4"
assert (ADAPTER / "adapter_model.safetensors").is_file(), f"PJME adapter missing: {ADAPTER}"
HORIZON, CONTEXT, BATCH = 24, 512, 32
val = series.loc[(series.index >= train_end) & (series.index < validation_end)]
train_values = series.loc[series.index < train_end].to_numpy(dtype=float)
scale = float(np.mean(np.abs(train_values[24:] - train_values[:-24])))
starts = np.arange(0, len(val) - HORIZON + 1, HORIZON, dtype=int)
contexts, actual_windows, target_times, origin_times = [], [], [], []
for offset in starts:
    origin = val.index[offset]
    absolute = series.index.get_loc(origin)
    context = series.iloc[absolute-CONTEXT:absolute].to_numpy(dtype=np.float32)
    if len(context) != CONTEXT:
        continue
    contexts.append(context)
    actual_windows.append(val.iloc[offset:offset+HORIZON].to_numpy(dtype=float))
    target_times.append(val.index[offset:offset+HORIZON])
    origin_times.append(origin)
contexts = np.stack(contexts)
actual_windows = np.stack(actual_windows)
print({"validation_windows":len(contexts), "hours":actual_windows.size, "first_target":str(target_times[0][0]), "last_target":str(target_times[-1][-1]), "context":CONTEXT, "horizon":HORIZON, "training_mase_scale":scale, "adapter":str(ADAPTER)})
predictions = {"weekly_seasonal_naive":np.stack([seasonal_naive(c,HORIZON) for c in contexts])}
zero_model = TimesFMZeroShot(CONTEXT, HORIZON, model_id="google/timesfm-2.5-200m-transformers")
predictions["timesfm_zero_shot"] = zero_model.predict_many(contexts, batch_size=BATCH)
del zero_model
gc.collect(); torch.cuda.empty_cache()
adapter_model = TimesFMZeroShot(CONTEXT, HORIZON, adapter=str(ADAPTER), model_id="google/timesfm-2.5-200m-transformers")
predictions["timesfm_pjme_adapter"] = adapter_model.predict_many(contexts, batch_size=BATCH)
del adapter_model
gc.collect(); torch.cuda.empty_cache()
summary = []
for name, forecast in predictions.items():
    row = {"model":name, "evaluation_split":"validation", "metrics":score(actual_windows.reshape(-1), forecast.reshape(-1), scale), "horizon_hours":HORIZON, "context_length":CONTEXT, "windows_evaluated":len(contexts), "hours_evaluated":actual_windows.size}
    summary.append(row)
naive_mae = summary[0]["metrics"]["MAE"]
for row in summary:
    row["MAE_skill_vs_weekly_naive"] = 1.0 - row["metrics"]["MAE"] / naive_mae
out_dir = Path("/kaggle/working/germany-transfer-validation")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir/"metrics.json").write_text(json.dumps(summary, indent=2)+"\n")
prediction_rows=[]
for model_name, forecast in predictions.items():
    for wi, origin in enumerate(origin_times):
        for lead, (timestamp, actual, estimate) in enumerate(zip(target_times[wi], actual_windows[wi], forecast[wi])):
            prediction_rows.append({"model":model_name,"window_start":origin,"timestamp":timestamp,"lead_hour":lead,"actual":float(actual),"forecast":float(estimate)})
all_predictions=pd.DataFrame(prediction_rows)
all_predictions.to_csv(out_dir/"predictions.csv", index=False)
pd.DataFrame([{"model":r["model"],**r["metrics"],"MAE_skill_vs_weekly_naive":r["MAE_skill_vs_weekly_naive"],"windows_evaluated":r["windows_evaluated"],"hours_evaluated":r["hours_evaluated"]} for r in summary]).to_csv(out_dir/"summary.csv",index=False)
lead_rows=[]
for model_name, group in all_predictions.groupby("model", sort=False):
    error=group["forecast"].to_numpy()-group["actual"].to_numpy()
    group=group.assign(abs_error=np.abs(error),sq_error=error**2)
    for lead, part in group.groupby("lead_hour",sort=True):
        lead_rows.append({"model":model_name,"lead_hour":int(lead),"MAE":float(part.abs_error.mean()),"RMSE":float(np.sqrt(part.sq_error.mean())),"hours":len(part)})
pd.DataFrame(lead_rows).to_csv(out_dir/"lead_metrics.csv",index=False)
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(12,4))
actual=all_predictions[(all_predictions.model=="weekly_seasonal_naive") & (all_predictions.window_start==origin_times[0])]
ax.plot(actual.timestamp,actual.actual,label="Actual",linewidth=2)
for name in predictions:
    part=all_predictions[(all_predictions.model==name)&(all_predictions.window_start==origin_times[0])]
    ax.plot(part.timestamp,part.forecast,label=name,linewidth=1.4)
ax.set(title="Germany validation: first 24-hour forecast",ylabel="Load (MW)",xlabel="UTC")
ax.legend(); fig.tight_layout(); fig.savefig(out_dir/"comparison.png",dpi=160); plt.close(fig)
print(json.dumps(summary,indent=2)); print("Saved",out_dir)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 21.8 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


Cloning into '/kaggle/working/energy-demand-timesfm'...


{'validation_windows': 365, 'hours': 8760, 'first_target': '2019-01-01 00:00:00+00:00', 'last_target': '2019-12-31 23:00:00+00:00', 'context': 512, 'horizon': 24, 'training_mase_scale': 4509.288670091324, 'adapter': '/kaggle/working/energy-demand-timesfm/outputs/timesfm-lora-pjme-t4'}


config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  925MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[
  {
    "model": "weekly_seasonal_naive",
    "evaluation_split": "validation",
    "metrics": {
      "MAE": 2558.145662100457,
      "RMSE": 4464.972134961397,
      "MASE": 0.5673058101309465
    },
    "horizon_hours": 24,
    "context_length": 512,
    "windows_evaluated": 365,
    "hours_evaluated": 8760,
    "MAE_skill_vs_weekly_naive": 0.0
  },
  {
    "model": "timesfm_zero_shot",
    "evaluation_split": "validation",
    "metrics": {
      "MAE": 3217.866262218179,
      "RMSE": 4737.1951398050505,
      "MASE": 0.7136083976085322
    },
    "horizon_hours": 24,
    "context_length": 512,
    "windows_evaluated": 365,
    "hours_evaluated": 8760,
    "MAE_skill_vs_weekly_naive": -0.25789016235144135
  },
  {
    "model": "timesfm_pjme_adapter",
    "evaluation_split": "validation",
    "metrics": {
      "MAE": 1926.302130600742,
      "RMSE": 2997.0785375902146,
      "MASE": 0.4271853659264022
    },
    "horizon_hours": 24,
    "context_length": 512,
    "windows_evaluat

In [4]:
%pip install --upgrade torchao
import torchao, torch, peft
print({'torch': torch.__version__, 'torchao': torchao.__version__, 'peft': peft.__version__})

Note: you may need to restart the kernel to use updated packages.
{'torch': '2.11.0+cu130', 'torchao': '0.18.0', 'peft': '0.21.2'}


In [5]:
import subprocess, sys
TRAIN_ADAPTER = REPO / "outputs/timesfm-lora-germany-t4"
cmd = [sys.executable, "-u", str(REPO / "scripts/finetune_lora.py"),
       "--data", "/kaggle/working/germany_hourly_load.csv",
       "--train-end", "2019-01-01T00:00:00Z",
       "--validation-end", "2020-01-01T00:00:00Z",
       "--context-len", "512", "--horizon-len", "24",
       "--epochs", "5", "--batch-size", "8", "--lr", "5e-5",
       "--lora-r", "4", "--lora-alpha", "8", "--samples", "2000",
       "--output-dir", str(TRAIN_ADAPTER), "--seed", "42", "--patience", "2"]
print('Training Germany LoRA on 2015-01-01 through 2018-12-31; validation 2019 only; test remains untouched.', flush=True)
subprocess.run(cmd, check=True)

Training Germany LoRA on 2015-01-01 through 2018-12-31; validation 2019 only; test remains untouched.


usage: finetune_lora.py [-h] --data DATA [--model-id MODEL_ID]
                        [--context-len CONTEXT_LEN]
                        [--horizon-len HORIZON_LEN] [--epochs EPOCHS]
                        [--batch-size BATCH_SIZE] [--lr LR] [--lora-r LORA_R]
                        [--lora-alpha LORA_ALPHA] [--samples SAMPLES]
                        [--output-dir OUTPUT_DIR] [--seed SEED]
                        [--resume-adapter RESUME_ADAPTER]
                        [--base-model-name BASE_MODEL_NAME]
                        [--patience PATIENCE]
finetune_lora.py: error: unrecognized arguments: --train-end 2019-01-01T00:00:00Z --validation-end 2020-01-01T00:00:00Z


CalledProcessError: Command '['/usr/bin/python3', '-u', '/kaggle/working/energy-demand-timesfm/scripts/finetune_lora.py', '--data', '/kaggle/working/germany_hourly_load.csv', '--train-end', '2019-01-01T00:00:00Z', '--validation-end', '2020-01-01T00:00:00Z', '--context-len', '512', '--horizon-len', '24', '--epochs', '5', '--batch-size', '8', '--lr', '5e-5', '--lora-r', '4', '--lora-alpha', '8', '--samples', '2000', '--output-dir', '/kaggle/working/energy-demand-timesfm/outputs/timesfm-lora-germany-t4', '--seed', '42', '--patience', '2']' returned non-zero exit status 2.

In [ ]:
from pathlib import Path
import json, gc
import numpy as np
import pandas as pd
import torch
from src.models import TimesFMZeroShot, seasonal_naive
from src.metrics import score

HORIZON, CONTEXT, BATCH = 24, 512, 32
TEST_DIR = Path("/kaggle/working/germany-transfer-test")
TEST_DIR.mkdir(parents=True, exist_ok=True)
test_series = series.loc[series.index >= validation_end]
training_values = series.loc[series.index < train_end].to_numpy(dtype=float)
mase_scale = float(np.mean(np.abs(training_values[24:] - training_values[:-24])))
test_starts = np.arange(0, len(test_series) - HORIZON + 1, HORIZON, dtype=int)
test_contexts, test_actual, test_times, test_origins = [], [], [], []
for offset in test_starts:
    origin = test_series.index[offset]
    absolute = series.index.get_loc(origin)
    context = series.iloc[absolute-CONTEXT:absolute].to_numpy(dtype=np.float32)
    if len(context) != CONTEXT:
        continue
    test_contexts.append(context)
    test_actual.append(test_series.iloc[offset:offset+HORIZON].to_numpy(dtype=float))
    test_times.append(test_series.index[offset:offset+HORIZON])
    test_origins.append(origin)
test_contexts = np.stack(test_contexts)
test_actual = np.stack(test_actual)
print({"test_windows":len(test_contexts),"hours":test_actual.size,
       "first_target":str(test_times[0][0]),"last_target":str(test_times[-1][-1]),
       "context":CONTEXT,"horizon":HORIZON,"train_mase_scale_mw":mase_scale}, flush=True)

test_predictions = {
    "weekly_seasonal_naive": np.stack([seasonal_naive(c,HORIZON) for c in test_contexts])
}
zero_model = TimesFMZeroShot(CONTEXT,HORIZON,model_id="google/timesfm-2.5-200m-transformers")
test_predictions["timesfm_zero_shot"] = zero_model.predict_many(test_contexts,batch_size=BATCH)
del zero_model
gc.collect(); torch.cuda.empty_cache()
pjme_model = TimesFMZeroShot(CONTEXT,HORIZON,adapter=str(ADAPTER),
                             model_id="google/timesfm-2.5-200m-transformers")
test_predictions["timesfm_pjme_adapter"] = pjme_model.predict_many(test_contexts,batch_size=BATCH)
del pjme_model
gc.collect(); torch.cuda.empty_cache()
germany_model = TimesFMZeroShot(CONTEXT,HORIZON,adapter=str(TRAIN_ADAPTER),
                                model_id="google/timesfm-2.5-200m-transformers")
test_predictions["timesfm_germany_adapter"] = germany_model.predict_many(test_contexts,batch_size=BATCH)
del germany_model
gc.collect(); torch.cuda.empty_cache()

actual_flat = test_actual.reshape(-1)
summary_test=[]
for name, forecast in test_predictions.items():
    row={"model":name,"evaluation_split":"test",
         "metrics":score(actual_flat,forecast.reshape(-1),mase_scale),
         "horizon_hours":HORIZON,"context_length":CONTEXT,
         "windows_evaluated":len(test_contexts),"hours_evaluated":test_actual.size,
         "first_target":str(test_times[0][0]),"last_target":str(test_times[-1][-1])}
    summary_test.append(row)
naive_mae=summary_test[0]["metrics"]["MAE"]
for row in summary_test:
    row["MAE_skill_vs_weekly_naive"]=1-row["metrics"]["MAE"]/naive_mae

prediction_rows=[]
for model_name, forecast in test_predictions.items():
    for wi, origin in enumerate(test_origins):
        for lead,(timestamp,actual,estimate) in enumerate(zip(test_times[wi],test_actual[wi],forecast[wi])):
            prediction_rows.append({"model":model_name,"window_start":origin,"timestamp":timestamp,
                                    "lead_hour":lead,"actual":float(actual),"forecast":float(estimate)})
all_test_predictions=pd.DataFrame(prediction_rows)
all_test_predictions.to_csv(TEST_DIR/"predictions.csv",index=False)
(TEST_DIR/"metrics.json").write_text(json.dumps(summary_test,indent=2)+"\n")
pd.DataFrame([{"model":r["model"],**r["metrics"],
               "MAE_skill_vs_weekly_naive":r["MAE_skill_vs_weekly_naive"],
               "windows_evaluated":r["windows_evaluated"],"hours_evaluated":r["hours_evaluated"]}
              for r in summary_test]).to_csv(TEST_DIR/"summary.csv",index=False)

lead_rows=[]
window_rows=[]
for model_name, group in all_test_predictions.groupby("model",sort=False):
    err=group["forecast"].to_numpy()-group["actual"].to_numpy()
    group=group.assign(abs_error=np.abs(err),sq_error=err**2)
    for lead,part in group.groupby("lead_hour",sort=True):
        lead_rows.append({"model":model_name,"lead_hour":int(lead),
                          "MAE":float(part.abs_error.mean()),
                          "RMSE":float(np.sqrt(part.sq_error.mean())),"hours":len(part)})
    for origin,part in group.groupby("window_start",sort=True):
        window_rows.append({"model":model_name,"window_start":origin,
                            "MAE":float(part.abs_error.mean()),
                            "RMSE":float(np.sqrt(part.sq_error.mean()))})
pd.DataFrame(lead_rows).to_csv(TEST_DIR/"lead_metrics.csv",index=False)
window_metrics=pd.DataFrame(window_rows)
window_metrics.to_csv(TEST_DIR/"daily_window_metrics.csv",index=False)
daily_wide=window_metrics.pivot(index="window_start",columns="model",values="MAE").reset_index()
daily_wide["germany_adapter_minus_weekly_naive_MAE"]=daily_wide["timesfm_germany_adapter"]-daily_wide["weekly_seasonal_naive"]
daily_wide["absolute_difference_mw"]=daily_wide["germany_adapter_minus_weekly_naive_MAE"].abs()
daily_wide.sort_values("absolute_difference_mw",ascending=False).head(20).to_csv(TEST_DIR/"largest_daily_differences.csv",index=False)

import matplotlib.pyplot as plt
first_origin=test_origins[0]
fig,ax=plt.subplots(figsize=(12,4))
for model_name,forecast in test_predictions.items():
    ax.plot(test_times[0],forecast[0],label=model_name,linewidth=1.3)
ax.plot(test_times[0],test_actual[0],label="Actual",linewidth=2,color="black")
ax.set(title="Germany test: first 24-hour forecast",ylabel="Load (MW)",xlabel="UTC")
ax.legend(); fig.tight_layout(); fig.savefig(TEST_DIR/"comparison_first_day.png",dpi=160); plt.close(fig)
fig,ax=plt.subplots(figsize=(12,4))
for model_name,part in window_metrics.groupby("model",sort=False):
    ax.plot(pd.to_datetime(part.window_start),part.MAE,label=model_name,linewidth=1)
ax.set(title="Germany test: daily-window MAE",ylabel="MAE (MW)",xlabel="Forecast start (UTC)")
ax.legend(); fig.tight_layout(); fig.savefig(TEST_DIR/"daily_mae.png",dpi=160); plt.close(fig)
print(json.dumps(summary_test,indent=2))
print("Saved final test artifacts to",TEST_DIR)

In [ ]:
# Descriptive test error review; no test-driven tuning.
window_metrics["MAE"] = pd.to_numeric(window_metrics["MAE"], errors="coerce")
daily_numeric = window_metrics.pivot(index="window_start",columns="model",values="MAE")
delta = (daily_numeric["timesfm_germany_adapter"] - daily_numeric["weekly_seasonal_naive"]).astype(float)
print("Germany adapter lower daily MAE than naive:", f"{(delta < 0).mean():.1%}")
print("Germany-minus-naive daily MAE min/median/max MW:",
      round(float(delta.min()),1),round(float(delta.median()),1),round(float(delta.max()),1))
print("Largest improvements:",[(str(pd.Timestamp(i).date()),round(float(v),1)) for i,v in delta.nsmallest(5).items()])
print("Largest regressions:",[(str(pd.Timestamp(i).date()),round(float(v),1)) for i,v in delta.nlargest(5).items()])
for model_name,part in window_metrics.groupby("model",sort=False):
    part=part.sort_values("MAE",ascending=False)
    print({"model":model_name,"median_daily_MAE_MW":round(float(part.MAE.median()),1),
           "p90_daily_MAE_MW":round(float(part.MAE.quantile(.9)),1),
           "worst_5_day_share":round(float(part.head(5).MAE.sum()/part.MAE.sum()),4),
           "worst_3_days":[(str(pd.Timestamp(r.window_start).date()),round(float(r.MAE),1))
                           for r in part.head(3).itertuples()]})
lead_table=pd.read_csv(TEST_DIR/"lead_metrics.csv")
print("Worst lead:",[(name,int(g.loc[g.MAE.idxmax(),"lead_hour"]),round(float(g.MAE.max()),1))
                      for name,g in lead_table.groupby("model",sort=False)])

# Bundle the series, selected Germany adapter, and all validation/test tables and figures.
import zipfile
from IPython.display import FileLink, display
ZIP_PATH=Path("/kaggle/working/germany-transfer-results.zip")
manifest={"dataset":"OPSD DE_load_actual_entsoe_transparency, 2020-10-06",
          "units":"MW","timezone":"UTC",
          "splits":{"train":"2015-01-01/2019-01-01","validation":"2019-01-01/2020-01-01",
                    "test":"2020-01-01/2020-10-01"},
          "context_hours":512,"horizon_hours":24,"selected_germany_checkpoint_epoch":5,
          "selected_germany_validation_loss":0.061289,
          "test_metrics":summary_test}
with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write("/kaggle/working/germany_hourly_load.csv","data/germany_hourly_load.csv")
    for folder,label in [(Path("/kaggle/working/germany-transfer-validation"),"validation"),
                         (TEST_DIR,"test"),
                         (TRAIN_ADAPTER,"germany_adapter")]:
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(path,f"{label}/{path.relative_to(folder)}")
    archive.writestr("manifest.json",json.dumps(manifest,indent=2)+"\n")
print({"results_bundle":str(ZIP_PATH),"bytes":ZIP_PATH.stat().st_size})
display(FileLink(str(ZIP_PATH)))

In [ ]:
# Date-based split boundaries are implemented in src/data.py and scripts/finetune_lora.py.print("The repository includes exclusive timestamp-boundary options for training and evaluation.")